# MCQ Intent-Based Similarity Search
**Fine-tune `paraphrase-MiniLM-L6-v2` on MMLU → compute pairwise similarity**

### Adaptive Answer Weight
`answer_weight = 0.15 + 0.70 × (1 − question_sim)`
- High q_sim → answer weight LOW (answer just confirms)
- Low q_sim, but same answer → answer weight HIGH (answer reveals hidden concept match)
- **Same answer ≠ Same intent**

In [ ]:
# Cell 1: Install
!pip install -q sentence-transformers datasets scikit-learn pandas openpyxl


In [ ]:
# # Cell 2: Imports & Config
# import json, os, time
# import numpy as np
# import pandas as pd
# from itertools import combinations
# from datasets import load_dataset, Dataset
# from sentence_transformers import (
#     SentenceTransformer,
#     SentenceTransformerTrainer,
#     SentenceTransformerTrainingArguments,
#     losses,
# )
# from sentence_transformers.evaluation import BinaryClassificationEvaluator
# from sklearn.metrics.pairwise import cosine_similarity
# import torch

# # ── Config (mirrors main.py DEFAULT_CONFIG) ──
# BASE_MODEL      = 'distilbert-base-uncased'
# QUORA_DATASET   = 'sentence-transformers/quora-duplicates'
# QUORA_SUBSET    = 'pair-class'   # ~404 K pairs with binary labels
# MAX_SAMPLES     = 50000          # reduce for faster testing
# EPOCHS          = 5
# BATCH_SIZE      = 32
# LEARNING_RATE   = 2e-5
# MARGIN          = 0.5            # OnlineContrastiveLoss margin
# SEED            = 42
# CUTOFF          = 0.75           # similarity threshold for validation
# MODEL_SAVE      = '/content/mcq_intent_model'
# print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')


In [ ]:
# Cell 2: Imports & Config
import json, os, time
import numpy as np
import pandas as pd
from itertools import combinations
from datasets import load_dataset, Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import BinaryClassificationEvaluator
from sklearn.metrics.pairwise import cosine_similarity
import torch

# ── Config (mirrors main.py DEFAULT_CONFIG) ──
# Updated to use the optimized sentence-transformer model
BASE_MODEL      = 'all-MiniLM-L6-v2'
QUORA_DATASET   = 'sentence-transformers/quora-duplicates'
QUORA_SUBSET    = 'pair-class'   # ~404 K pairs with binary labels
MAX_SAMPLES     = 50000          # reduce for faster testing
EPOCHS          = 5
BATCH_SIZE      = 32
LEARNING_RATE   = 2e-5           # May want to lower to 1e-5 for fine-tuning
MARGIN          = 0.5            # OnlineContrastiveLoss margin
SEED            = 42
CUTOFF          = 0.75           # similarity threshold for validation
# Updated save path to reflect the new model
MODEL_SAVE      = '/content/mcq_intent_model_minilm'
print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

Device: CUDA


In [ ]:
# Cell 3: Load Quora Duplicates Dataset
print('Loading Quora Duplicates dataset...')
raw = load_dataset(QUORA_DATASET, QUORA_SUBSET, split='train')

# Standardise column names
if 'sentence1' in raw.column_names:   # pair-class schema
    raw = raw.rename_columns({'sentence1': 'anchor', 'sentence2': 'positive'})

# Sub-sample
if MAX_SAMPLES and len(raw) > MAX_SAMPLES:
    raw = raw.shuffle(seed=SEED).select(range(MAX_SAMPLES))

# Train / eval split
split = raw.train_test_split(test_size=0.1, seed=SEED)
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}')


Loading Quora Duplicates dataset...


README.md: 0.00B [00:00, ?B/s]

pair-class/train-00000-of-00001.parquet:   0%|          | 0.00/35.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/404290 [00:00<?, ? examples/s]

Train: 45,000  |  Eval: 5,000


In [ ]:
# Cell 4: Fine-tune with SentenceTransformerTrainer + OnlineContrastiveLoss
import os

model = SentenceTransformer(BASE_MODEL)

# Loss: OnlineContrastiveLoss works well for binary similarity tasks
loss_fn = losses.OnlineContrastiveLoss(model=model, margin=MARGIN)

# Evaluator: Binary classification metrics on eval set
evaluator = BinaryClassificationEvaluator(
    sentences1=eval_ds['anchor'],
    sentences2=eval_ds['positive'],
    labels=eval_ds['label'],
    name='quora-eval',
    batch_size=BATCH_SIZE,
)

# Training arguments  (use_mps_device removed – not supported in current ST)
args = SentenceTransformerTrainingArguments(
    output_dir=MODEL_SAVE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),   # enable on GPU, disable on CPU/MPS
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
   metric_for_best_model='eval_quora-eval_cosine_ap',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    seed=SEED,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=loss_fn,
    evaluator=evaluator,
)

print(f'Fine-tuning {EPOCHS} epoch(s) on {len(train_ds):,} pairs...')
t0 = time.time()
trainer.train()
model.save(MODEL_SAVE)
print(f'Done in {time.time()-t0:.0f}s  ->  saved to {MODEL_SAVE}')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done in 558s  ->  saved to /content/mcq_intent_model_minilm


In [ ]:
# Cell 5: Evaluate & calibrate similarity threshold
from sklearn.metrics import precision_recall_fscore_support

model = SentenceTransformer(MODEL_SAVE)
anchors   = eval_ds['anchor']
positives = eval_ds['positive']
labels    = eval_ds['label']

emb_a = model.encode(anchors,   batch_size=64, show_progress_bar=True, convert_to_numpy=True)
emb_p = model.encode(positives, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
sims  = (emb_a * emb_p).sum(axis=1) / (
    np.linalg.norm(emb_a, axis=1) * np.linalg.norm(emb_p, axis=1) + 1e-9
)

best_thresh, best_f1 = CUTOFF, 0.0
for t in np.arange(0.50, 0.95, 0.05):
    preds = (sims >= t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, round(float(t), 2)

print(f'Optimal threshold: {best_thresh:.2f}  (F1={best_f1:.4f})')
CUTOFF = best_thresh


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Optimal threshold: 0.80  (F1=0.8264)


In [ ]:
# Cell 4b: Zip and Download Model Weights
import shutil
from google.colab import files

# 1. Zip the saved model directory
shutil.make_archive('/content/mcq_intent_model', 'zip', MODEL_SAVE)
print('Model zipped successfully: /content/mcq_intent_model.zip')

# 2. Download to your local machine
files.download('/content/mcq_intent_model.zip')

# Optional: Copy to Google Drive (Uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/mcq_intent_model.zip "/content/drive/MyDrive/"
# print('Saved to Google Drive!')


Model zipped successfully: /content/mcq_intent_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 6: Set path to your MCQ JSON file
import os

# ✏️ Change this to your actual file path
mcq_filename = '/content/english.json'   # e.g. '/content/drive/MyDrive/mcqs.json'

assert os.path.exists(mcq_filename), f"File not found: {mcq_filename}"
print(f"✅ Using MCQ file: {mcq_filename}")


AssertionError: File not found: /content/english.json

In [ ]:
# Cell 7: Load MCQs
# Format: [{"question": str, "options": [str, ...], "answer": str}, ...]
with open(mcq_filename, "r", encoding="utf-8") as f:
    raw = json.load(f)

mcqs = []
for m in raw:
    q       = (m.get("question") or "").strip()
    options = m.get("options", [])          # list of option strings
    answer  = str(m.get("answer") or "").strip()

    if not q or not options or not answer:
        continue

    # Find the index of the correct answer in options list
    answer_idx = next(
        (i for i, opt in enumerate(options) if str(opt).strip() == answer),
        None
    )
    if answer_idx is None:
        continue   # skip if answer does not match any option

    mcqs.append({
        "question"  : q,
        "options"   : [str(o).strip() for o in options],
        "answer"    : answer,
        "answer_idx": answer_idx,
    })

print(f"Loaded {len(mcqs)} MCQs  (skipped {len(raw) - len(mcqs)} invalid)")
for m in mcqs[:3]:
    print(f"  Q: {m['question']}")
    print(f"  A[{m['answer_idx']}]: {m['answer']}  |  opts: {m['options']}")
    print()


In [ ]:
# Cell 8: Encode Questions & Answers
model = SentenceTransformer(MODEL_SAVE)
questions = [m['question'] for m in mcqs]
answers   = [m['answer']   for m in mcqs]
print('Encoding questions...')
q_embs = model.encode(questions, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
print('Encoding answers...')
a_embs = model.encode(answers, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
q_mat = cosine_similarity(q_embs, q_embs)
a_mat = cosine_similarity(a_embs, a_embs)
print(f'Matrix shape: {q_mat.shape}')

In [ ]:
# Cell 9: Adaptive-Weight Pairwise Similarity
# answer_weight = 0.15 + 0.70 * (1 - q_sim)
# High q_sim -> low ans weight (answer barely adds info)
# Low q_sim  -> high ans weight (answer reveals the hidden match)
results = []
n = len(mcqs)
for i, j in combinations(range(n), 2):
    q_sim = float(q_mat[i][j])
    a_sim = float(a_mat[i][j])
    answer_weight   = 0.15 + 0.70 * (1.0 - q_sim)
    question_weight = 1.0 - answer_weight
    final_sim = answer_weight * a_sim + question_weight * q_sim
    if final_sim < CUTOFF:
        continue
    label = 'Duplicate' if final_sim >= 0.90 else 'Highly Similar' if final_sim >= 0.80 else 'Similar'
    results.append({
        'q1': mcqs[i]['question'], 'a1': mcqs[i]['answer'],
        'q2': mcqs[j]['question'], 'a2': mcqs[j]['answer'],
        'question_sim': round(q_sim, 4),
        'answer_sim':   round(a_sim, 4),
        'answer_weight': round(answer_weight, 4),
        'final_sim':    round(final_sim, 4),
        'label':        label,
    })
results.sort(key=lambda x: x['final_sim'], reverse=True)
print(f'Found {len(results)} pairs | Duplicates: {sum(1 for r in results if r["label"]=="Duplicate")}')

In [ ]:
# Cell 10: Show Results Table
df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 45)
df[['label','q1','a1','q2','a2','question_sim','answer_sim','answer_weight','final_sim']].head(20)

In [ ]:
# Cell 11: Print Top Pairs
for r in results[:15]:
    print(f"\n[{r['label']}]")
    print(f"  Q1: {r['q1']}  ->  A1: {r['a1']}")
    print(f"  Q2: {r['q2']}  ->  A2: {r['a2']}")
    print(f"  q_sim={r['question_sim']:.4f}  a_sim={r['answer_sim']:.4f}  "
          f"ans_weight={r['answer_weight']:.4f}  final={r['final_sim']:.4f}")

In [ ]:
# Cell 13: Evaluate on test.csv — Question-Only Similarity
# No answer weighting; pure cosine similarity between question1 and question2
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# ── 1. Load model ──────────────────────────────────────────────────────────
model = SentenceTransformer(MODEL_SAVE)
print(f'Loaded model from: {MODEL_SAVE}')

# ── 2. Load test CSV ───────────────────────────────────────────────────────
TEST_CSV = '/content/test.csv'   # ✏️ change path if needed
df_test = pd.read_csv(TEST_CSV)
print(f'Test set: {len(df_test):,} rows')
print(df_test.head(3))

# Normalise column names to lowercase
df_test.columns = [c.lower() for c in df_test.columns]
assert 'question1' in df_test.columns and 'question2' in df_test.columns, \
    'CSV must have columns: question1 and question2'

q1_list = df_test['question1'].fillna('').tolist()
q2_list = df_test['question2'].fillna('').tolist()

# ── 3. Encode both question columns ───────────────────────────────────────
print('\nEncoding question1...')
emb1 = model.encode(q1_list, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print('Encoding question2...')
emb2 = model.encode(q2_list, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

# ── 4. Row-wise cosine similarity (question-only, NO answer weight) ────────
# Embeddings are L2-normalised, so dot-product == cosine similarity
sims = (emb1 * emb2).sum(axis=1).astype(float)   # shape: (N,)

# ── 5. Apply CUTOFF threshold & assign human-readable labels ──────────────
preds = (sims >= CUTOFF).astype(int)   # 1 = similar, 0 = not similar

def label_sim(s):
    if s >= 0.90:    return 'Duplicate'
    if s >= 0.80:    return 'Highly Similar'
    if s >= CUTOFF:  return 'Similar'
    return 'Not Similar'

df_test['similarity_score'] = np.round(sims, 4)
df_test['predicted_label']  = [label_sim(s) for s in sims]
df_test['is_similar_pred']  = preds   # 1 / 0

# ── 6. If ground-truth labels exist, compute full metrics ─────────────────
label_col = next(
    (c for c in df_test.columns if c in ('is_duplicate', 'label', 'similarity')),
    None
)
if label_col:
    true = df_test[label_col].astype(int).values
    acc  = accuracy_score(true, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        true, preds, average='binary', zero_division=0
    )
    print(f'\n=== Evaluation Metrics (threshold = {CUTOFF}) ===')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print('\nDetailed Report:')
    print(classification_report(true, preds, target_names=['Not Duplicate', 'Duplicate']))

    # Grid-search for best threshold
    print('\n--- Threshold Sweep ---')
    best_thresh, best_f1 = CUTOFF, 0.0
    for t in np.arange(0.45, 0.96, 0.05):
        p = (sims >= t).astype(int)
        _, _, fi, _ = precision_recall_fscore_support(true, p, average='binary', zero_division=0)
        print(f'  threshold={t:.2f}  F1={fi:.4f}')
        if fi > best_f1:
            best_f1, best_thresh = fi, round(float(t), 2)
    print(f'\nBest threshold: {best_thresh}  (F1={best_f1:.4f})')
else:
    print('\n(No ground-truth label column found — showing predictions only)')

# ── 7. Show sample results ─────────────────────────────────────────────────
pd.set_option('display.max_colwidth', 55)
display(df_test[['question1', 'question2', 'similarity_score', 'predicted_label']].head(20))

# ── 8. Prediction distribution ────────────────────────────────────────────
print('\n=== Prediction Distribution ===')
print(df_test['predicted_label'].value_counts().to_string())

# ── 9. Export predictions CSV ─────────────────────────────────────────────
out_path = '/content/test_predictions.csv'
df_test.to_csv(out_path, index=False)
print(f'\nSaved predictions → {out_path}')
try:
    from google.colab import files
    files.download(out_path)
except Exception:
    pass


In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

MODEL_SAVE = '/home/cpatwadityasharma/Downloads/question_genration/similarity/mcq_intent_model_dir'
CUTOFF = 0.80

print("Loading model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
model = SentenceTransformer(MODEL_SAVE, device=device)
if device == 'cuda':
    model.half()  # Use FP16 computation on GPU for massive speed boost and lower VRAM
print(f'Loaded model from: {MODEL_SAVE}')

TEST_CSV = '/home/cpatwadityasharma/Downloads/question_genration/similarity/test.csv'
df_test = pd.read_csv(TEST_CSV)
print(f'Test set: {len(df_test):,} rows')

q1_list = df_test['question1'].fillna('').tolist()
q2_list = df_test['question2'].fillna('').tolist()

print('\nEncoding question1...')
emb1 = model.encode(q1_list, batch_size=1024, show_progress_bar=True, normalize_embeddings=True)
print('Encoding question2...')
emb2 = model.encode(q2_list, batch_size=1024, show_progress_bar=True, normalize_embeddings=True)

sims = (emb1 * emb2).sum(axis=1).astype(float)
preds = (sims >= CUTOFF).astype(int)

def label_sim(s):
    if s >= 0.90:    return 'Duplicate'
    if s >= 0.80:    return 'Highly Similar'
    if s >= CUTOFF:  return 'Similar'
    return 'Not Similar'

df_test['similarity_score'] = np.round(sims, 4)
df_test['predicted_label']  = [label_sim(s) for s in sims]
df_test['is_similar_pred']  = preds

label_col = next((c for c in df_test.columns if c in ('is_duplicate', 'label', 'similarity')), None)
if label_col:
    print(f"Found true label column: {label_col}")
    true = df_test[label_col].astype(int).values
    acc  = accuracy_score(true, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true, preds, average='binary', zero_division=0)
    print(f'\n=== Evaluation Metrics (threshold = {CUTOFF}) ===')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print('\nDetailed Report:')
    print(classification_report(true, preds, target_names=['Not Duplicate', 'Duplicate']))
else:
    print('\n(No ground-truth label column found in df_test — showing predictions only)')

submission = pd.DataFrame()
submission['test_id'] = df_test['test_id']

# Output similarities clipped to [0,1] range as probabilities
submission['is_duplicate'] = np.clip(sims, 0, 1)

out_path = '/home/cpatwadityasharma/Downloads/question_genration/similarity/submission.csv'
submission.to_csv(out_path, index=False)
print(f'\nSaved predictions -> {out_path}')


Loading model...
Using device: cuda


FileNotFoundError: Path /home/cpatwadityasharma/Downloads/question_genration/similarity/mcq_intent_model_dir not found

In [ ]:
!unzip -r "/Users/k/rag_questions/similiarty /test.csv.zip"